In [7]:
import pandas as pd
import numpy as np
import folium

In [6]:
path_relativo = '../data/processed/'
datos = pd.read_csv(path_relativo + 'andaluces_2_5k.csv')
hospitales = pd.read_csv(path_relativo + 'Hospitales_Completo.csv')
display(hospitales)

,nombre,localidad,provincia,capacidad,latitud,longitud
0,Hospital Virgen del Mar,Almería,Almería,76,36.856070,-2.417615
1,Hospital Universitario Torrecardenas,Almería,Almería,779,36.862595,-2.441142
2,Hospital La Inmaculada,Huércal-Overa,Almería,184,37.401403,-1.942036
3,Hospital Universitario de Poniente,"Ejido, El",Almería,281,36.753477,-2.803836
4,Hospital Mediterráneo,Almería,Almería,89,36.820367,-2.435320
...,...,...,...,...,...,...
128,Hospital Psiquiátrico Penitenciario,Sevilla,Sevilla,184,37.390565,-5.847516
129,Clínica de Salud Mental Miguel de Mañara,Dos Hermanas,Sevilla,18,37.334785,-5.918881
130,Hospital de La Mujer,Sevilla,Sevilla,221,37.363119,-5.978213
131,"Adinfa, Sociedad Cooperativa Andaluza",Coria del Río,Sevilla,25,37.287022,-6.056425


In [8]:
def create_map(datos: pd.DataFrame, hospitales: pd.DataFrame, out_html: str):
    # Asegurar que latitud y longitud son numéricas
    datos['latitud'] = pd.to_numeric(datos['latitud'], errors='coerce')
    datos['longitud'] = pd.to_numeric(datos['longitud'], errors='coerce')

    hospitales['latitud'] = pd.to_numeric(hospitales['latitud'], errors='coerce')
    hospitales['longitud'] = pd.to_numeric(hospitales['longitud'], errors='coerce')

    # Centro del mapa = media de coordenadas de municipios (o conjunta)
    center_lat = datos['latitud'].mean()
    center_lon = datos['longitud'].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=7)

    # --- Municipios en azul ---
    for _, row in datos.iterrows():
        if pd.isna(row['latitud']) or pd.isna(row['longitud']):
            continue
        popup_text = (
            f"<b>{row['municipio']}</b><br>"
            f"Provincia: {row['PROVINCIA']}<br>"
            f"Población: {int(row['poblacion'])}<br>"
            f"Lat, Lon: {row['latitud']:.5f}, {row['longitud']:.5f}"
        )
        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=4,
            popup=folium.Popup(popup_text, max_width=250),
            fill=True,
            color='blue',
            fill_color='blue',
            fill_opacity=0.6
        ).add_to(m)

    # --- Hospitales en rojo ---
    for _, row in hospitales.iterrows():
        if pd.isna(row['latitud']) or pd.isna(row['longitud']):
            continue
        popup_text = (
            f"<b>{row['nombre']}</b><br>"
            f"Localidad: {row['localidad']}<br>"
            f"Provincia: {row['provincia']}<br>"
            f"Lat, Lon: {row['latitud']:.5f}, {row['longitud']:.5f}"
        )
        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=6,
            popup=folium.Popup(popup_text, max_width=250),
            fill=True,
            color='red',
            fill_color='red',
            fill_opacity=0.9
        ).add_to(m)

    m.save(out_html)
    return m


In [9]:
m=create_map(datos, hospitales, '../maps/raw/mapa_andalucia_prueba.html')